# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [20]:
%load_ext dotenv
%dotenv ../05_src/.secrets

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [11]:
from langchain_community.document_loaders import PyPDFLoader

file_path = r"C:\Users\Jaswin\Desktop\DSI\deploying-ai\Managing Oneself_Drucker_HBR.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load()

print(len(docs))

13


In [12]:
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [13]:
from openai import OpenAI
client= OpenAI()
Model="gpt-4o-mini"


In [14]:
# Make the model write concise Summary
def summarize(text: str) -> str:
    # Keep prompt short and clear; truncate very long texts to limit cost
    chunk = text[:12000]  # ~ safe chunk for a short summary
    user_prompt = f"""
You are a clear, neutral summarizer for beginners.
Summarize the article below with:
- 1-sentence main thesis
- 3 bullet points of key ideas
- 1 practical takeaway
Avoid quotes; use your own words.

ARTICLE:
{chunk}
"""
    resp = client.chat.completions.create(
        model=Model,
        messages=[{"role":"user","content":user_prompt}],
        temperature=0.2,
    )
    return resp.choices[0].message.content.strip()

summary = summarize(document_text)
print(summary)

**Main Thesis:** Success in today's knowledge economy requires individuals to take responsibility for their own careers by understanding their strengths, values, and optimal work environments.

**Key Ideas:**
- Knowledge workers must act as their own chief executive officers, managing their careers and making informed decisions about their professional paths.
- Self-awareness is crucial; individuals should identify their strengths and weaknesses through methods like feedback analysis to enhance their performance and contributions.
- Aligning personal values with organizational ethics is essential for job satisfaction and effectiveness, as a mismatch can lead to frustration and poor performance.

**Practical Takeaway:** Regularly practice feedback analysis by documenting your expectations for key decisions and comparing them to actual outcomes to better understand your strengths and areas for improvement.


In [15]:
from pydantic import BaseModel,Field,ValidationError
from typing import List
import json, textwrap # to make sure textwrap is imported

#Asking Model to evaluate the summary and force valide JSON(so grader can parse it)
class Scores(BaseModel):
    faithfulness: int = Field(..., ge=1, le=5, description="No hallucinations; grounded in the source")
    coverage: int    = Field(..., ge=1, le=5, description="Captures main thesis and major points")
    coherence: int   = Field(..., ge=1, le=5, description="Well-structured and easy to follow")
    concision: int   = Field(..., ge=1, le=5, description="Brief without losing substance")

class EvalResult(BaseModel):
    scores: Scores
    evidence: List[str] = Field(default_factory=list, description="Short quotes/phrases from source")
    verdict: str = Field(..., pattern="^(pass|fail)$")
    comments: str

EVAL_INSTRUCTIONS = """
You are a meticulous evaluator of a summary.
Rubric (1=poor, 5=excellent):
- Faithfulness: No contradictions; facts align with the article.
- Coverage: Main thesis + most important points are present.
- Coherence: Clear structure; flows logically.
- Concision: No fluff; succinct.

Return STRICT JSON with keys: scores{{faithfulness,coverage,coherence,concision}}, evidence[], verdict("pass" or "fail"), comments.
Passing rule: average score >= 4 and no dimension < 3.
"""

eval_prompt = f"""{EVAL_INSTRUCTIONS}

ARTICLE (reference):
{textwrap.shorten(document_text, width=8000, placeholder=" ...")}

SUMMARY (to grade):
{summary}
"""

resp = client.chat.completions.create(
    model=Model,
    messages=[{"role":"user","content":eval_prompt}],
    # JSON mode: model must reply with a valid JSON object
    response_format={"type":"json_object"},
    temperature=0,  # deterministic grading
)
raw_json = resp.choices[0].message.content
print(raw_json)

try:
    eval_obj = EvalResult.model_validate_json(raw_json)
    avg = (eval_obj.scores.faithfulness + eval_obj.scores.coverage +
           eval_obj.scores.coherence + eval_obj.scores.concision) / 4
    print(f"\nAverage score: {avg:.2f}  |  Verdict: {eval_obj.verdict}")
    print("\nComments:", eval_obj.comments)
    print("\nEvidence:", eval_obj.evidence[:5])
except ValidationError as e:
    print("Validation error:", e)
    

{
  "scores": {
    "faithfulness": 5,
    "coverage": 5,
    "coherence": 5,
    "concision": 5
  },
  "evidence": [
    "The summary accurately reflects the main thesis of the article, emphasizing the need for self-awareness and personal responsibility in career management.",
    "All key ideas from the article are present, including the role of knowledge workers as their own CEOs, the importance of self-awareness, and the alignment of personal values with organizational ethics.",
    "The structure of the summary is clear, with a logical flow from the main thesis to key ideas and practical takeaways.",
    "The summary is succinct and free of unnecessary fluff, conveying the essential points effectively."
  ],
  "verdict": "pass",
  "comments": "The summary is comprehensive, accurate, and well-structured, meeting all criteria for a high-quality evaluation."
}

Average score: 5.00  |  Verdict: pass

Comments: The summary is comprehensive, accurate, and well-structured, meeting all cr

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [16]:
import deepeval

In [18]:
from deepeval.test_case import LLMTestCase
from deepeval.metrics import SummarizationMetric
from deepeval import evaluate

# 2) Sanity checks: you must already have these from earlier cells
if "document_text" not in globals() or not isinstance(document_text, str) or not document_text.strip():
    raise RuntimeError("`document_text` is missing. Load your article text into `document_text` first.")
if "summary" not in globals() or not isinstance(summary, str) or not summary.strip():
    raise RuntimeError("`summary` is missing. Run your summarize() step to set `summary` first.")

# 3) Create the REQUIRED test case object (this is what `tc` should be)
tc = LLMTestCase(
    input=document_text,        # the original article text
    actual_output=summary     # the model-produced summary you want to evaluate
)

# 4) Your custom metric with checklist questions
metric_custom = SummarizationMetric(
    threshold=0.75,
    model="gpt-4o-mini",  # optional; remove if your DeepEval version complains
    assessment_questions=[
        "Does the summary correctly state the main thesis?",
        "Does it include the most important finding/result?",
        "Does it avoid adding claims that are not present in the article?",
        "Does it mention key limitations/assumptions if applicable?",
        "Is the conclusion consistent with the article?"
    ],
    include_reason=True
)

# 5) Measure and inspect results
metric_custom.measure(tc)
print("Score:", round(metric_custom.score, 3))
print("Pass? ", metric_custom.score >= metric_custom.threshold)
print("Reason:\n", metric_custom.reason)
print("Breakdown:", getattr(metric_custom, "score_breakdown", None))  # alignment & coverage


Output()

Score: 0
Pass?  False
Reason:
 The score is 0.00 because the summary contains significant contradictions to the original text, specifically regarding the alignment of personal values with organizational ethics and the consequences of mismatches, which are not addressed in the original text.
Breakdown: {'Alignment': 0.75, 'Coverage': 0}


# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

enhancement_prompt = f"""
You previously wrote this summary in Victorian English:

{initial_summary}

Evaluation feedback:
- Summarization: {summarization_result.reason}
- Coherence: {coherence_result.reason}
- Tonality: {tonality_result.reason}
- Safety: {safety_result.reason}

Context:
{reference_text}

Task:
Revise the summary to address the feedback above while preserving the Victorian English tone.
Ensure:
- Accuracy and completeness of key ideas from the context.
- Improved clarity and logical flow.
- Maintain Victorian English style consistently.
- Keep the summary under 1000 tokens.
Return only the improved summary text.
"""
print(document_text)

In [19]:
from openai import OpenAI
client = OpenAI()
MODEL = "gpt-4o-mini"

# Reuse what you already have:
# - document_text (full article)  OR  article_excerpt = document_text[:8000]
# - summary (your first draft)
# - raw_json (JSON-mode evaluation)  OR  deepeval_reason (e.g., metric.reason)

article_excerpt = document_text[:8000]  # keep it short for cost/latency
evaluation_text = None

# Prefer your JSON-mode evaluation if you have it; else fall back to DeepEval reason
if "raw_json" in globals() and isinstance(raw_json, str) and raw_json.strip():
    evaluation_text = raw_json
elif "metric_custom" in globals() and getattr(metric_custom, "reason", None):
    evaluation_text = metric_custom.reason
else:
    evaluation_text = "No structured evaluation available."

prompt = f"""
You are an expert summarization editor. Improve a short summary using (A) the source context
and (B) evaluator feedback. Priorities: faithfulness and coverage, then coherence and concision.

INPUTS
CONTEXT (authoritative, prefer this over everything else):
<<<{article_excerpt}>>>

CURRENT_SUMMARY (to improve):
<<<{summary}>>>

EVALUATION_FEEDBACK (JSON or text; may include scores, reasons, or a checklist):
<<<{evaluation_text}>>>

TASK
1) Fix any faithfulness issues flagged in EVALUATION_FEEDBACK. Do not add facts not supported by CONTEXT.
2) Raise coverage: include the article’s main thesis and the 3–6 most important points
   (prioritize items EVALUATION_FEEDBACK says were missing/under-emphasized).
3) Improve coherence and concision.
4) If a checklist is present, ensure each item is addressed.

CONSTRAINTS
- Neutral tone. No direct quotes; paraphrase.
- Max 180–220 words total.

OUTPUT FORMAT (exact)
Summary:
- Thesis: <1 sentence>
- Key Points:
  - <1>
  - <2>
  - <3>
  - <4>
  - <5-6 optional>
- Practical Takeaway: <1 sentence>

Changes Made (brief):
- <…>
- <…>

Uncertainties / Open Questions (max 3, optional):
- <…>

QUALITY CHECK:
- Faithfulness: <yes/no>
- Coverage: <high/medium/low>
"""

resp = client.chat.completions.create(
    model=Model,
    messages=[{"role": "user", "content": prompt}],
    temperature=0.2
)
improved_summary = resp.choices[0].message.content
print(improved_summary)


Summary:
- Thesis: Success in the knowledge economy hinges on individuals taking charge of their careers through self-awareness of their strengths, values, and optimal work environments.
- Key Points:
  - Knowledge workers must act as their own chief executive officers, managing their careers and making informed decisions about their professional trajectories.
  - Self-awareness is vital; individuals should identify their strengths and weaknesses through feedback analysis to enhance performance and contributions.
  - Aligning personal values with organizational ethics is crucial for job satisfaction, as a mismatch can lead to frustration and decreased effectiveness.
  - To discover strengths, individuals should practice feedback analysis by documenting expected outcomes of key decisions and comparing them with actual results.
  - Understanding one’s preferred work style and environment is essential for maximizing contributions and achieving excellence.
- Practical Takeaway: Regularly e


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
